In [1]:
import json
import os

# ================= 配置区 =================
# 1. 评测结果 JSONL 文件路径
FILE_PATH = "/mnt/data/zwl/verl/lm-evaluation-harness/samples_aime24_base_2026-04-28T01-40-50.255191.jsonl"
# 2. 模型名称（将作为 output 下的子文件夹名）
MODEL_NAME = "qwen3-4b—sft"
# ==========================================

def write_markdown(filepath, q_id, target_ans, model_ans, raw_response, title_suffix):
    """辅助函数：用于统一格式生成 Markdown 文件"""
    with open(filepath, "w", encoding="utf-8") as out_f:
        out_f.write(f"# 题目 ID: {q_id}\n\n")
        out_f.write(f"**正确答案 (Target):** {target_ans}\n")
        out_f.write(f"**模型提取答案 (Model):** {model_ans}\n\n")
        out_f.write("---\n\n")
        out_f.write(f"## 模型完整生成的{title_suffix}：\n\n")
        out_f.write(raw_response)

def analyze_and_extract(jsonl_path, model_tag):
    if not os.path.exists(jsonl_path):
        print(f"找不到文件: {jsonl_path}")
        return

    # 获取当前执行脚本的目录，并创建 output/model_tag 层级
    # 这样所有文件都会集中存放，通过文件名开头的前缀区分
    current_dir = os.getcwd()
    save_dir = os.path.join(current_dir, "output", model_tag)
    os.makedirs(save_dir, exist_ok=True)

    results = []
    correct_count, incorrect_count, invalid_count = 0, 0, 0

    print(f"正在处理文件并导出至: {save_dir} ...\n")

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): 
                continue
            data = json.loads(line)
            
            q_id = data.get("doc", {}).get("ID", "Unknown")
            target_ans = data.get("target", "Unknown")
            filtered_resps = data.get("filtered_resps", ["[None]"])
            model_ans = filtered_resps[0] if filtered_resps else "[None]"
            raw_response = data.get("resps", [[""]])[0][0]
            
            is_correct = data.get("exact_match", 0.0) == 1.0
            is_invalid = "[invalid]" in filtered_resps or model_ans == "[invalid]"

            results.append({
                "ID": q_id, 
                "Correct": target_ans, 
                "Model": model_ans, 
                "IsCorrect": is_correct,
                "IsInvalid": is_invalid
            })

            # 根据状态决定文件名前缀
            if is_correct:
                correct_count += 1
                prefix = "正确"
                suffix = "正确解答过程"
            elif is_invalid:
                invalid_count += 1
                incorrect_count += 1
                prefix = "invalid"
                suffix = "解答过程 (格式错误/死循环)"
            else:
                incorrect_count += 1
                prefix = "错误"
                suffix = "【错误】解答过程"

            # 导出文件：例如 "正确_2024-II-1.md"
            file_name = f"{prefix}_{q_id}.md"
            out_file = os.path.join(save_dir, file_name)
            write_markdown(out_file, q_id, target_ans, model_ans, raw_response, suffix)

    # 打印汇总表格
    results.sort(key=lambda x: x["ID"])
    print(f"{'题目 ID':<15} | {'标准答案':<15} | {'模型答案':<15} | {'状态'}")
    print("-" * 75)
    
    for r in results:
        status = "✅ 正确" if r['IsCorrect'] else ("⚠️ Invalid" if r['IsInvalid'] else "❌ 错误")
        print(f"{r['ID']:<15} | {str(r['Correct'])[:15]:<15} | {str(r['Model'])[:15]:<15} | {status}")
        
    print(f"\n总计: {len(results)} 题 | 正确: {correct_count} | 错误: {incorrect_count} (含 {invalid_count} 个 invalid)")
    print(f"所有结果已保存至: {save_dir}")

if __name__ == "__main__":
    analyze_and_extract(FILE_PATH, MODEL_NAME)

正在处理文件并导出至: /mnt/workspace/zwl/verl/lm-evaluation-harness/output/qwen3-4b—sft ...

题目 ID           | 标准答案            | 模型答案            | 状态
---------------------------------------------------------------------------
2024-I-1        | 204             | 204             | ✅ 正确
2024-I-10       | 113             | 113             | ✅ 正确
2024-I-11       | 371             | 35              | ❌ 错误
2024-I-12       | 385             | 16              | ❌ 错误
2024-I-13       | 110             | 110             | ✅ 正确
2024-I-14       | 104             | 104             | ✅ 正确
2024-I-15       | 721             | 721             | ✅ 正确
2024-I-2        | 25              | 25              | ✅ 正确
2024-I-3        | 809             | 809             | ✅ 正确
2024-I-4        | 116             | 116             | ✅ 正确
2024-I-5        | 104             | 104             | ✅ 正确
2024-I-6        | 294             | 294             | ✅ 正确
2024-I-7        | 540             | 540             | ✅ 正确
2024-I-8        |